# Basic и Manual

Сравниваем два ранжировщика LightGBM для 500 кандидатов: Basic использует метаданные и сигналы генератора, Manual дополнительно использует историю просмотров и совпадения категорий.

Обучение на шести предыдущих неделях: 5000 пользователей и 500 кандидатов в каждой неделе. Оценка: семь недель с 21 июня по 2 августа 2021 года; тест 9 и 16 августа на моделях последнего фолда. Метрики: MAP@10 и NDCG@10.

In [1]:
import pandas as pd

interactions = pd.read_csv('data_original/interactions.csv', parse_dates=['last_watch_dt'])
users = pd.read_csv('data_original/users.csv')
items = pd.read_csv('data_original/items.csv')

interactions.head()

,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250,72.0
1,699317,1659,2021-05-29,8317,100.0
2,656683,7107,2021-05-09,10,0.0
3,864613,7638,2021-07-05,14483,100.0
4,964868,9506,2021-04-30,6725,100.0


In [2]:
users.head()

,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,М,1
1,962099,age_18_24,income_20_40,М,0
2,1047345,age_45_54,income_40_60,Ж,0
3,721985,age_45_54,income_20_40,Ж,0
4,704055,age_35_44,income_60_90,Ж,0


In [3]:
items.head()

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
2,10716,film,Тактическая сила,Tactical Force,2011.0,"криминал, зарубежные, триллеры, боевики, комедии",Канада,NaN,16.0,NaN,Адам П. Калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг..."
3,7868,film,45 лет,45 Years,2015.0,"драмы, зарубежные, мелодрамы",Великобритания,NaN,16.0,NaN,Эндрю Хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю..."
4,16268,film,Все решает мгновение,NaN,1978.0,"драмы, спорт, советские, мелодрамы",СССР,NaN,12.0,Ленфильм,Виктор Садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж..."


In [4]:
item_basic = items[["item_id", "content_type", "release_year", "age_rating", "for_kids"]].copy()
item_basic["content_type"] = item_basic["content_type"].astype("category")

for field, prefix in [("genres", "genre_"), ("countries", "country_"), ("studios", "studio_")]:
    encoded = items[field].fillna("").str.get_dummies(sep=", ")
    item_basic = pd.concat([item_basic, encoded.add_prefix(prefix).astype("uint8")], axis=1)

user_basic = users[["user_id", "age", "income", "sex", "kids_flg"]].copy()
for field in ("age", "income", "sex"):
    user_basic[field] = user_basic[field].astype("category")

retrieval_cols = ["pop_rank", "bm25_rank", "bm25_score", "rrf_score", "rrf_rank"]
basic_cols = retrieval_cols + user_basic.columns.drop("user_id").tolist() + item_basic.columns.drop("item_id").tolist()
display(item_basic.head())
display(user_basic.head())
print('Признаков Basic:', len(basic_cols))

,item_id,content_type,release_year,age_rating,for_kids,genre_18+,genre_no_genre,genre_анимация,genre_аниме,genre_артхаус,...,studio_Sony Pictures Television,studio_Sony Plus,studio_Starz,studio_Universal,studio_Warner Bros,studio_Warner Bros. Television,studio_Ленфильм,studio_Мосфильм,studio_Рок фильм,studio_рентв
0,10711,film,2002.0,16.0,NaN,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2508,film,2014.0,16.0,NaN,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,10716,film,2011.0,16.0,NaN,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,7868,film,2015.0,16.0,NaN,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,16268,film,1978.0,12.0,NaN,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,М,1
1,962099,age_18_24,income_20_40,М,0
2,1047345,age_45_54,income_40_60,Ж,0
3,721985,age_45_54,income_20_40,Ж,0
4,704055,age_35_44,income_60_90,Ж,0


Признаков Basic: 233


In [5]:
import numpy as np
from scipy import sparse
from sklearn.preprocessing import MultiLabelBinarizer

item_index = pd.Index(items["item_id"])

attrs = {}
for name in ("genre", "country", "studio"):
    cols = item_basic.columns[item_basic.columns.str.startswith(name + "_")]
    attrs[name] = sparse.csr_matrix(item_basic[cols].to_numpy(dtype=np.float32))

for name, column in [("actor", "actors"), ("director", "directors")]:
    lists = items[column].fillna("").map(lambda x: x.split(", ") if x else [])
    attrs[name] = MultiLabelBinarizer(sparse_output=True).fit_transform(lists).astype(np.float32)

pd.DataFrame({name: [*matrix.shape, matrix.nnz] for name, matrix in attrs.items()},
             index=['Объекты','Категории','Ненулевые значения']).T

,Объекты,Категории,Ненулевые значения
genre,15963,95,40637
country,15963,92,18295
studio,15963,33,16018
actor,15963,61887,139670
director,15963,8839,17810


In [6]:
def user_history_features(h, t):
    g = h.groupby("user_id")
    f = g.agg(user_n_items=("item_id", "size"),
              user_last_dt=("last_watch_dt", "max"),
              user_watched_pct_mean=("watched_pct", "mean"),
              user_total_dur_mean=("total_dur", "mean"))
    f["user_days_since_last"] = (t - f.pop("user_last_dt")).dt.days
    for days in (7, 30):
        f[f"user_n_items_{days}d"] = h.loc[h.last_watch_dt >= t - pd.Timedelta(days=days)].groupby("user_id").size()
    return f.fillna({"user_n_items_7d": 0, "user_n_items_30d": 0}).reset_index()

def item_history_features(h, t):
    f = h.groupby("item_id").agg(item_popularity=("user_id", "size"),
                                 item_first_dt=("last_watch_dt", "min"),
                                 item_watched_pct_mean=("watched_pct", "mean"))
    f["item_days_since_first"] = (t - f.pop("item_first_dt")).dt.days
    for days in (7, 30):
        recent = h.loc[h.last_watch_dt >= t - pd.Timedelta(days=days)]
        f[f"item_popularity_{days}d"] = recent.groupby("item_id").size()
        f[f"item_popularity_{days}d"] = f[f"item_popularity_{days}d"].fillna(0)
        f[f"item_pop_share_{days}d"] = f[f"item_popularity_{days}d"] / max(len(recent), 1)
    f["item_trend_7v30"] = (f["item_popularity_7d"] + 1) / (f["item_popularity_30d"] / 30 * 7 + 1)
    f["item_pop_share_all"] = f["item_popularity"] / max(len(h), 1)
    return f.reset_index()

In [7]:
history = interactions[interactions.last_watch_dt < '2021-06-21']
display(user_history_features(history, pd.Timestamp('2021-06-21')).head())
display(item_history_features(history, pd.Timestamp('2021-06-21')).head())
del history

,user_id,user_n_items,user_watched_pct_mean,user_total_dur_mean,user_days_since_last,user_n_items_7d,user_n_items_30d
0,2,43,81.627907,5418.860465,2,11.0,40.0
1,3,1,0.000000,142.000000,18,0.0,1.0
2,4,1,0.000000,14.000000,19,0.0,1.0
3,5,17,9.764706,442.058824,10,0.0,17.0
4,7,1,17.000000,672.000000,75,0.0,0.0


,item_id,item_popularity,item_watched_pct_mean,item_days_since_first,item_popularity_7d,item_pop_share_7d,item_popularity_30d,item_pop_share_30d,item_trend_7v30,item_pop_share_all
0,0,21,12.761905,82,1.0,0.000003,2.0,0.000002,1.363636,0.000009
1,1,14,65.857143,70,1.0,0.000003,4.0,0.000004,1.034483,0.000006
2,2,23,62.695652,99,0.0,0.000000,2.0,0.000002,0.681818,0.000010
3,3,12,71.416667,100,0.0,0.000000,3.0,0.000003,0.588235,0.000005
4,4,6,47.333333,97,0.0,0.000000,0.0,0.000000,1.000000,0.000003


In [8]:
def candidate_context(df, item_hist):
    pop7 = item_hist.set_index("item_id")["item_popularity_7d"]
    recent = df["item_id"].map(pop7).fillna(0)
    df["item_pop_rank500"] = recent.groupby(df["user_id"]).rank(ascending=False, method="average").astype("float32")
    df["item_pop_vs_candidates"] = (recent / (recent.groupby(df["user_id"]).transform("mean") + 1)).astype("float32")
    df["retrieval_n_sources"] = df["pop_rank"].notna().astype("uint8") + df["bm25_rank"].notna().astype("uint8")
    df["retrieval_rank_gap"] = (df["pop_rank"] - df["bm25_rank"]).astype("float32")
    return df

In [9]:
def user_item_matrix(h, users_idx):
    r = users_idx.get_indexer(h["user_id"])
    c = item_index.get_indexer(h["item_id"])
    return sparse.csr_matrix((np.ones(len(h), dtype=np.float32), (r, c)),
                             shape=(len(users_idx), len(item_index)))

def pair_dot(pairs, counts, attribute_matrix, users_idx, batch_size=100_000):
    u = users_idx.get_indexer(pairs["user_id"])
    i = item_index.get_indexer(pairs["item_id"])
    scores = np.empty(len(pairs), dtype=np.float32)
    for start in range(0, len(pairs), batch_size):
        stop = min(start + batch_size, len(pairs))
        scores[start:stop] = np.asarray(counts[u[start:stop]].multiply(attribute_matrix[i[start:stop]]).sum(axis=1)).ravel()
    return scores

def pair_overlap(pairs, A, users_idx, attribute_matrix):
    shared = np.zeros(len(pairs), dtype=np.int32)
    largest = np.zeros(len(pairs), dtype=np.float32)
    item_ids = pairs["item_id"].to_numpy()
    for user, pos in pairs.groupby("user_id", sort=False).indices.items():
        watched = A[users_idx.get_loc(user)].indices
        candidates = item_index.get_indexer(item_ids[pos])
        overlap = (attribute_matrix[watched] @ attribute_matrix[candidates].T).tocsr()
        shared[pos] = overlap.getnnz(axis=0)
        largest[pos] = overlap.max(axis=0).toarray().ravel()
    return shared, largest

In [10]:
def user_taste_features(h, item_hist):
    known = h[["user_id", "item_id"]].merge(
        items[["item_id", "release_year", "age_rating"]], on="item_id", how="left"
    )
    known["item_popularity"] = known["item_id"].map(item_hist.set_index("item_id")["item_popularity"])
    known["log_pop"] = np.log1p(known["item_popularity"])
    return known.groupby("user_id").agg(
        user_mean_release_year=("release_year", "mean"),
        user_mean_age_rating=("age_rating", "mean"),
        user_mean_log_pop=("log_pop", "mean")
    ).reset_index()

def add_pair_distances(df):
    df["release_year_gap"] = (df["release_year"] - df["user_mean_release_year"]).abs()
    df["age_rating_gap"] = (df["age_rating"] - df["user_mean_age_rating"]).abs()
    df["popularity_taste_gap"] = np.log1p(df["item_popularity"]) - df["user_mean_log_pop"]
    return df

In [11]:
def week_history(t):
    h = interactions.loc[interactions['last_watch_dt'] < t]
    item_hist = item_history_features(h, t)
    by_user = h.set_index('user_id', drop=False).sort_index()
    return item_hist, by_user, by_user.index.unique()

def user_context(t, user_ids, item_hist, by_user, known_users):
    requested = pd.Index(np.unique(user_ids))
    found = requested.intersection(known_users, sort=False)
    h = by_user.loc[found].reset_index(drop=True)
    idx = pd.Index(h['user_id'].unique())
    A = user_item_matrix(h, idx)
    A30 = user_item_matrix(h.loc[h['last_watch_dt'] >= t - pd.Timedelta(days=30)], idx)
    counts = {name: (A @ B, A30 @ B) for name, B in attrs.items()}
    return dict(item_hist=item_hist, user_hist=user_history_features(h, t),
                taste=user_taste_features(h, item_hist), users_idx=idx,
                A=A, A30=A30, counts=counts)

In [12]:
def relational_from_context(df, ctx):
    A, A30, idx = ctx["A"], ctx["A30"], ctx["users_idx"]
    u = idx.get_indexer(df["user_id"])
    n = A.getnnz(axis=1)[u]
    n30 = A30.getnnz(axis=1)[u]

    out = pd.DataFrame(index=df.index)
    for name, B in attrs.items():
        freq, freq30 = ctx["counts"][name]
        match = pair_dot(df, freq, B, idx)
        out[f"{name}_match_sum"] = match
        if name == "genre":
            idf = np.log((len(item_index) + 1) / (B.getnnz(axis=0) + 1))
            out["genre_match_idf"] = pair_dot(df, freq.multiply(idf).tocsr(), B, idx) / n
        out[f"{name}_match_mean"] = match / n
        if name in ("genre", "actor", "director"):
            shared, largest = pair_overlap(df, A, idx, B)
            out[f"{name}_shared_items"] = shared
            out[f"{name}_shared_fraction"] = shared / n
            out[f"{name}_match_max"] = largest
            out[f"{name}_match_30d"] = pair_dot(df, freq30, B, idx) / np.maximum(n30, 1)
    return out

In [13]:
def prepare_val_batch(raw, ctx):
    df = candidate_context(raw.reset_index(drop=True).copy(), ctx["item_hist"])
    df = df.merge(user_basic, on="user_id", how="left", validate="many_to_one")
    df = df.merge(item_basic, on="item_id", how="left", validate="many_to_one")
    df = df.merge(ctx["user_hist"], on="user_id", how="left", validate="many_to_one")
    df = df.merge(ctx["item_hist"], on="item_id", how="left", validate="many_to_one")
    df = df.merge(ctx["taste"], on="user_id", how="left", validate="many_to_one")
    df = add_pair_distances(df)
    rel = relational_from_context(df[["user_id", "item_id"]], ctx)
    df = pd.concat([df.reset_index(drop=True), rel.reset_index(drop=True)], axis=1)
    numeric = df.select_dtypes(include=["float64", "int64"]).columns.difference(["user_id", "item_id", "label"])
    df[numeric] = df[numeric].astype("float32")
    return df

In [14]:
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
from tqdm.auto import tqdm
import gc

replay_dir = Path('data_processed/replay')
work_dir = Path('data_processed/kion_weekly_5000_full500')
result_dir = Path('results/lgbm_cv_weekly5000_full500_sliding6')
for path in (work_dir / 'replay', work_dir / 'features', work_dir / 'sampled_users', result_dir):
    path.mkdir(parents=True, exist_ok=True)

/home/mpodsytnik/.conda/envs/avito/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
def week_user_ids(t):
    path = work_dir / 'sampled_users' / f'{t:%Y-%m-%d}.npy'
    if path.exists():
        return np.load(path)

    ids = pd.read_parquet(replay_dir / f'{t:%Y-%m-%d}.parquet', columns=['user_id'])['user_id'].to_numpy()
    groups = ids.reshape(-1, 500)
    available = groups[:, 0]
    n = min(5000, len(available))
    seed = 42 + (pd.Timestamp(t) - pd.Timestamp('2021-01-01')).days
    selected = np.random.default_rng(seed).choice(available, n, replace=False)
    np.save(path, selected)
    return selected

def sampled_replay(t):
    destination = work_dir / 'replay' / f'{t:%Y-%m-%d}.parquet'
    if destination.exists():
        return destination

    source = replay_dir / destination.name
    parquet = pq.ParquetFile(source)
    chosen = week_user_ids(t)
    chosen_arrow = pa.array(chosen, type=parquet.schema_arrow.field('user_id').type)
    temporary = destination.with_suffix('.tmp.parquet')
    writer = pq.ParquetWriter(temporary, parquet.schema_arrow, compression='zstd')
    try:
        for batch in parquet.iter_batches(batch_size=250_000):
            table = pa.Table.from_batches([batch])
            kept = table.filter(pc.is_in(table['user_id'], value_set=chosen_arrow))
            if len(kept):
                writer.write_table(kept)
    except BaseException:
        writer.close()
        temporary.unlink(missing_ok=True)
        raise
    writer.close()
    temporary.replace(destination)
    return destination

In [16]:
def train_dates_for(t):
    return pd.date_range(t - pd.Timedelta(weeks=6),
                         periods=6, freq='7D')

def prepare_train_week(t):
    destination = work_dir / 'features' / f'{t:%Y-%m-%d}.parquet'
    if destination.exists():
        return destination

    raw = pd.read_parquet(sampled_replay(t))
    raw['prediction_date'] = pd.to_datetime(raw['prediction_date'])
    n_users = len(raw) // 500
    item_hist, by_user, known_users = week_history(t)
    temporary = destination.with_suffix('.tmp.parquet')
    writer = None
    try:
        for start in tqdm(range(0, n_users, 1000), desc=f'Prepare {t:%Y-%m-%d}'):
            part = raw.iloc[start * 500:(start + 1000) * 500].reset_index(drop=True)
            ctx = user_context(t, part['user_id'].iloc[::500].to_numpy(),
                               item_hist, by_user, known_users)
            featured = prepare_val_batch(part, ctx)
            table = pa.Table.from_pandas(featured, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporary, table.schema, compression='zstd')
            writer.write_table(table)
            del ctx, part, featured, table
    except BaseException:
        if writer is not None:
            writer.close()
        temporary.unlink(missing_ok=True)
        raise
    writer.close()
    temporary.replace(destination)
    del raw, item_hist, by_user, known_users
    gc.collect()
    return destination

In [17]:
example = pq.ParquetFile(prepare_train_week(pd.Timestamp('2021-05-10')))
print('Строк подготовленной недели:', example.metadata.num_rows)
example = next(example.iter_batches(batch_size=5)).to_pandas()
display(example[['user_id','item_id','label','pop_rank','bm25_rank','user_n_items','item_popularity_7d','genre_match_idf']].head())

Строк подготовленной недели: 2500000


,user_id,item_id,label,pop_rank,bm25_rank,user_n_items,item_popularity_7d,genre_match_idf
0,46,9996,0,10.0,109.0,11.0,3639.0,0.686018
1,46,1884,0,93.0,17.0,11.0,562.0,0.686018
2,46,5658,0,8.0,164.0,11.0,784.0,0.705116
3,46,4151,0,1.0,305.0,11.0,6039.0,0.387253
4,46,10838,0,157.0,11.0,11.0,432.0,0.711991


In [18]:
def load_train_weeks(dates):
    frames = []
    for t in tqdm(dates, desc='Load train'):
        frames.append(pd.read_parquet(prepare_train_week(t)))
    train = pd.concat(frames, ignore_index=True)
    del frames
    gc.collect()
    return train

In [19]:
def ranking_metrics(scores, labels, target_n, rrf_ranks):
    target_n = np.asarray(target_n, dtype=np.int32)
    n = len(target_n)
    scores = np.asarray(scores).reshape(n, 500)
    ranks = np.asarray(rrf_ranks).reshape(n, 500)
    top = np.lexsort((ranks, -scores), axis=1)[:, :10]
    rel = np.take_along_axis(labels.reshape(n, 500), top, axis=1)
    precision = np.cumsum(rel, axis=1) / np.arange(1, 11)
    ap = (precision * rel).sum(axis=1) / np.minimum(target_n, 10)
    discounts = 1 / np.log2(np.arange(2, 12))
    dcg = (rel * discounts).sum(axis=1)
    ideal = np.r_[0, np.cumsum(discounts)][np.minimum(target_n, 10)]
    return {'map10': float(ap.mean()), 'ndcg10': float((dcg / ideal).mean())}

In [20]:
import lightgbm as lgb

def fit_ranker(train, valid, columns, group_train, group_valid):
    model = lgb.LGBMRanker(
        objective='lambdarank', n_estimators=500, learning_rate=0.05,
        num_leaves=31, max_bin=127, random_state=42, verbosity=-1,
        lambdarank_truncation_level=13, force_col_wise=True, histogram_pool_size=2048
    )
    model.fit(
        train[columns], train['label'], group=group_train,
        eval_set=[(valid[columns], valid['label'])],
        eval_group=[group_valid], eval_at=[10],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(50)],
    )
    return model

In [21]:
def target_counts_for_week(t):
    future = interactions.loc[
        (interactions['last_watch_dt'] >= t) &
        (interactions['last_watch_dt'] < t + pd.Timedelta(days=7))
    ]
    return future.groupby('user_id')['item_id'].nunique()

In [22]:
def fit_fold_models(t, train, basic_cols, all_cols):
    models = {}
    missing = []
    for name, cols in [('Basic', basic_cols), ('Manual', all_cols)]:
        path = result_dir / f'{name.lower()}_{t:%Y%m%d}.txt'
        if path.exists():
            try:
                models[name] = lgb.Booster(model_file=str(path)).num_trees()
                print(f'{name}: модель уже есть ({models[name]} деревьев)')
                continue
            except Exception:
                path.unlink()
        missing.append((name, cols))
    if not missing:
        return models

    es = pd.read_parquet(prepare_train_week(t))
    gt = train.groupby(['prediction_date', 'user_id'], sort=False).size().to_numpy()
    gv = es.groupby('user_id', sort=False).size().to_numpy()
    for name, cols in missing:
        print(f'Обучение {name} · {t:%Y-%m-%d}')
        model = fit_ranker(train, es, cols, gt, gv)
        models[name] = model.best_iteration_
        path = result_dir / f'{name.lower()}_{t:%Y%m%d}.txt'
        temporary = path.with_suffix('.tmp.txt')
        model.booster_.save_model(str(temporary), num_iteration=model.best_iteration_)
        temporary.replace(path)
        del model
        gc.collect()
    del es, gt, gv
    gc.collect()
    return models

In [23]:
def replay_batches(t):
    parquet = pq.ParquetFile(replay_dir / f'{t:%Y-%m-%d}.parquet')
    remainder = None
    for batch in parquet.iter_batches(batch_size=1000 * 500):
        part = batch.to_pandas()
        if remainder is not None:
            part = pd.concat([remainder, part], ignore_index=True)
        stop = (len(part) // 500) * 500
        if stop:
            ready = part.iloc[:stop].reset_index(drop=True)
            yield ready
        remainder = part.iloc[stop:].copy() if stop < len(part) else None

In [24]:
def score_fold(t, iterations, basic_cols, all_cols, model_date=None):
    boosters = {
        name: lgb.Booster(model_file=str(result_dir / f'{name.lower()}_{(model_date or t):%Y%m%d}.txt'))
        for name in ('Basic', 'Manual')
    }
    target_n = target_counts_for_week(t)
    item_hist, by_user, known_users = week_history(t)
    totals = {name: {'map10': 0.0, 'ndcg10': 0.0} for name in ('RRF', 'Basic', 'Manual')}
    groups = 0

    for raw in tqdm(replay_batches(t), desc=f'Validation {t:%Y-%m-%d}'):
        n = len(raw) // 500
        batch_users = raw['user_id'].iloc[::500]
        counts = target_n.reindex(batch_users).to_numpy()
        counts = counts.astype(np.int32)
        ctx = user_context(t, batch_users.to_numpy(), item_hist, by_user, known_users)
        featured = prepare_val_batch(raw, ctx)
        labels = featured['label'].to_numpy().reshape(-1, 500)
        rrf = featured['rrf_rank'].to_numpy().reshape(-1, 500)
        scores = {
            'RRF': -rrf,
            'Basic': boosters['Basic'].predict(featured[basic_cols]),
            'Manual': boosters['Manual'].predict(featured[all_cols]),
        }
        for name, values in scores.items():
            metrics = ranking_metrics(values, labels, counts, rrf)
            for key in ('map10', 'ndcg10'):
                totals[name][key] += metrics[key] * n
        groups += n
        del featured, raw, ctx, labels, rrf, scores

    del item_hist, by_user, known_users, boosters
    gc.collect()
    return [dict(week=str(t.date()), model=name, val_groups=groups,
                 train_weeks=6, train_users_per_week=5000,
                 best_iteration=iterations.get(name),
                 **{key: totals[name][key] / groups for key in ('map10', 'ndcg10')})
            for name in ('RRF', 'Basic', 'Manual')]

In [25]:
example_week = train_dates_for(pd.Timestamp('2021-06-21'))[0]
example = next(pq.ParquetFile(prepare_train_week(example_week)).iter_batches(batch_size=5)).to_pandas()
manual_cols = [c for c in example.columns if c not in {'user_id', 'item_id', 'label', 'prediction_date'} and c not in basic_cols]
all_cols = basic_cols + manual_cols
print('Признаков Basic:', len(basic_cols), '| Manual:', len(all_cols))

Признаков Basic: 233 | Manual: 281


In [26]:
cv_dates = pd.date_range('2021-06-21', periods=7, freq='7D')

cv_file = result_dir / 'cv_results.csv'
results = pd.read_csv(cv_file).to_dict('records') if cv_file.exists() else []
done = ({week for week, part in pd.DataFrame(results).groupby('week')
         if {'RRF', 'Basic', 'Manual'}.issubset(set(part['model']))} if results else set())

for t in cv_dates:
    if str(t.date()) in done:
        print(f'{t.date()}: готово — пропускаем')
        continue
    dates = train_dates_for(t)
    train = load_train_weeks(dates)
    groups = train.groupby(['prediction_date', 'user_id'], sort=False).size().to_numpy()
    print(f'Фолд {t.date()}: {len(train):,} строк; {len(groups):,} групп; '
          f'{train.memory_usage(deep=True).sum()/2**30:.1f} GiB')

    iterations = fit_fold_models(t, train, basic_cols, all_cols)
    del train, groups
    gc.collect()

    fold_result = score_fold(t, iterations, basic_cols, all_cols)
    results = [r for r in results if r['week'] != str(t.date())] + fold_result
    temporary = cv_file.with_suffix('.tmp')
    pd.DataFrame(results).to_csv(temporary, index=False)
    temporary.replace(cv_file)
    print(pd.DataFrame(fold_result).to_string(index=False))
    gc.collect()

display(pd.DataFrame(results).pivot(index='week', columns='model', values='map10').round(5))

Load train: 100%|██████████| 6/6 [02:35<00:00, 25.93s/it]


Фолд 2021-06-21: 15,000,000 строк; 30,000 групп; 6.9 GiB


Prepare 2021-06-21: 100%|██████████| 5/5 [00:24<00:00,  4.89s/it]


Обучение Basic · 2021-06-21


/home/mpodsytnik/.conda/envs/avito/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


...

/home/mpodsytnik/.conda/envs/avito/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[50]	valid_0's ndcg@10: 0.340768
[100]	valid_0's ndcg@10: 0.341223
[150]	valid_0's ndcg@10: 0.343285


Validation 2021-08-02: 100it [06:30,  3.90s/it]

      week  model  val_groups  train_weeks  train_users_per_week  best_iteration    map10   ndcg10
2021-08-02    RRF       99400            6                  5000             NaN 0.054630 0.087914
2021-08-02  Basic       99400            6                  5000           129.0 0.109496 0.155606
2021-08-02 Manual       99400            6                  5000           126.0 0.117743 0.169023


model,Basic,Manual,RRF
week,,,
2021-06-21,0.14912,0.16610,0.06084
2021-06-28,0.19703,0.25916,0.06214
2021-07-05,0.16865,0.13092,0.06038
2021-07-12,0.11767,0.14214,0.05332
2021-07-19,0.11112,0.12076,0.05294
2021-07-26,0.11225,0.11993,0.05434
2021-08-02,0.10950,0.11774,0.05463


In [28]:
test_file = result_dir / 'test_results.csv'
test_results = pd.read_csv(test_file).to_dict('records') if test_file.exists() else []
final_date = cv_dates[-1]
test_dates = pd.to_datetime([
    "2021-08-09",
    "2021-08-16"
])

for t in test_dates:
    if str(t.date()) in {r['week'] for r in test_results}:
        continue
    print('Тест:', t.date())
    rows = score_fold(t, {'Basic': None, 'Manual': None}, basic_cols, all_cols,
                      model_date=final_date)
    test_results = [r for r in test_results if r['week'] != str(t.date())] + rows
    pd.DataFrame(test_results).to_csv(test_file, index=False)

Тест: 2021-08-09


Validation 2021-08-09: 106it [06:55,  3.92s/it]


Тест: 2021-08-16


Validation 2021-08-16: 112it [07:22,  3.95s/it]


In [29]:
for filename in ('cv_results.csv', 'test_results.csv'):
    result = pd.read_csv(result_dir / filename)
    print(filename)
    display(result.pivot(index='week', columns='model', values='map10').round(5))
    display(result.groupby('model')[['map10', 'ndcg10']].mean().round(5))

model = lgb.Booster(model_file=str(result_dir / f'manual_{cv_dates[-1]:%Y%m%d}.txt'))
importance = pd.Series(model.feature_importance(importance_type='gain'),
                       index=model.feature_name()).sort_values(ascending=False)
display((importance / importance.sum()).head(15).round(3))

cv_results.csv


model,Basic,Manual,RRF
week,,,
2021-06-21,0.14912,0.16610,0.06084
2021-06-28,0.19703,0.25916,0.06214
2021-07-05,0.16865,0.13092,0.06038
2021-07-12,0.11767,0.14214,0.05332
2021-07-19,0.11112,0.12076,0.05294
2021-07-26,0.11225,0.11993,0.05434
2021-08-02,0.10950,0.11774,0.05463


,map10,ndcg10
model,,
Basic,0.13791,0.19278
Manual,0.15096,0.21121
RRF,0.05694,0.09488


test_results.csv


model,Basic,Manual,RRF
week,,,
2021-08-09,0.09548,0.10669,0.04999
2021-08-16,0.09019,0.10426,0.04760


,map10,ndcg10
model,,
Basic,0.09284,0.13314
Manual,0.10548,0.15231
RRF,0.04880,0.08066


item_pop_vs_candidates    0.378
item_popularity_7d        0.186
bm25_score                0.083
rrf_rank                  0.082
item_pop_rank500          0.081
item_pop_share_7d         0.030
item_trend_7v30           0.027
genre_мелодрамы           0.011
item_watched_pct_mean     0.010
country_match_mean        0.009
actor_match_mean          0.007
user_mean_log_pop         0.007
genre_match_idf           0.007
rrf_score                 0.007
bm25_rank                 0.006
dtype: float64